# Aula ML 01 — Introdução à Machine Learning

Objetivo: olhar as **medidas** de um pinguim e prever a **espécie**.

Rode as células **em ordem**, de cima para baixo (Shift+Enter).


## 1) Abrir a tabela

Vamos ler o arquivo CSV. Ele vira uma tabela no Python.


In [ ]:
import pandas as pd

pinguins = pd.read_csv("dados/penguins_size.csv")
pinguins.head(4)


## 2) Ver as espécies

Quantos pinguins de cada espécie existem? (só nas linhas sem buraco nas medidas)


In [ ]:
# nomes das colunas de medida
medidas = [
    "culmen_length_mm",   # comprimento do bico
    "culmen_depth_mm",    # profundidade do bico
    "flipper_length_mm",  # comprimento da asa
    "body_mass_g",        # massa
]

# tira linhas com valor faltando (só para olhar)
completos = pinguins.dropna(subset=medidas + ["species"])
print(completos["species"].value_counts())


## 3) Um gráfico simples

Cada ponto é um pinguim. Eixo X = bico. Eixo Y = asa. Cor = espécie.


In [ ]:
import matplotlib.pyplot as plt

cores = {"Adelie": "tab:blue", "Gentoo": "tab:orange", "Chinstrap": "tab:green"}

fig, ax = plt.subplots()
for especie, cor in cores.items():
    pedaco = completos[completos["species"] == especie]
    ax.scatter(
        pedaco["culmen_length_mm"],
        pedaco["flipper_length_mm"],
        s=20,
        c=cor,
        label=especie,
    )

ax.set_xlabel("bico (mm)")
ax.set_ylabel("asa (mm)")
ax.legend()
plt.show()


## 4) Limpar os dados

Algumas linhas têm buraco (NaN). Nesta aula, **apagamos** essas linhas.

Depois separamos:
- o que **entra** no modelo = as 4 medidas
- o que queremos **prever** = a espécie


In [ ]:
print("linhas no arquivo:", len(pinguins))

pinguins = pinguins.dropna(subset=medidas + ["species"])
print("linhas depois de limpar:", len(pinguins))

# o que entra
X = pinguins[medidas]
# o que queremos prever
y = pinguins["species"]

print("colunas que entram:", list(X.columns))
print("exemplo de respostas:", y.head().tolist())


## 5) Separar treino e teste

- **Treino** = o modelo estuda nestes exemplos
- **Teste** = a prova (o modelo não estuda aqui)

Usamos cerca de 3/4 no treino e 1/4 no teste.


In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.25,      # 25% para a prova
    stratify=y,          # mantém a mistura das espécies
    random_state=0,      # para todo mundo ter o mesmo sorteio
)

print("treino:", len(X_treino))
print("teste :", len(X_teste))


## 6) Treinar o modelo

Usamos **regressão logística** (um classificador).

Antes, igualamos a escala das medidas (massa é em milhares; bico é em dezenas).
Os dois passos ficam juntos no `Pipeline`.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modelo = Pipeline([
    ("escala", StandardScaler()),
    ("classificador", LogisticRegression(max_iter=1000)),
])

# aprende só com o treino
modelo.fit(X_treino, y_treino)
print("treino concluído")


## 7) Prever um pinguim da prova

Pegamos **1** pinguim do teste. O modelo só vê as medidas e chuta a espécie.


In [ ]:
# primeira linha do teste
um_pinguim = X_teste.iloc[[0]]

print("medidas:")
print(um_pinguim)

real = y_teste.iloc[0]
previsto = modelo.predict(um_pinguim)[0]

print("espécie real    :", real)
print("espécie prevista:", previsto)


## 8) Nota da prova (acurácia)

Acurácia = **quantos acertamos** no teste ÷ **quantos tinha** no teste.


In [ ]:
previstos = modelo.predict(X_teste)
acertos = (previstos == y_teste).sum()
total = len(y_teste)

print(acertos, "acertos de", total)
print("acurácia:", round(acertos / total, 3))


## Pronto

Você fechou o ciclo: ler → limpar → separar → treinar → prever → medir.

Se algo der erro, rode de novo **desde a primeira célula**.
